# Imports and bucket setup

In [ ]:
!wb resource mount

In [ ]:
import os
import re
# import pandas as pd
# import numpy as np

try:
    from google.cloud import bigquery
except ImportError:
    bigquery = None

In [ ]:
import os
import subprocess

cmd = """
source /home/jupyter/load-env.sh >/dev/null
env
"""

result = subprocess.run(
    ["bash", "-lc", cmd],
    capture_output=True,
    text=True,
    check=True,
)

for line in result.stdout.splitlines():
    if "=" in line:
        key, value = line.split("=", 1)
        os.environ[key] = value

print("WORKSPACE_CDR:", os.environ.get("WORKSPACE_CDR"))
print("WORKSPACE_BUCKET:", os.environ.get("WORKSPACE_BUCKET"))
os.environ["WORKSPACE_CDR"] = "wb-silky-artichoke-2408.C2025Q4R6"

In [ ]:
bucket = os.getenv("WORKSPACE_BUCKET")
cdr = os.environ.get("WORKSPACE_CDR")

if cdr is None:
    raise EnvironmentError(
        "WORKSPACE_CDR is not set. This script should be run inside an All of Us workspace."
    )

use_bqstorage = ("BIGQUERY_STORAGE_API_ENABLED" in os.environ)
use_bqstorage

# Helpers

In [ ]:
def run_gbq_query(sql: str, label: str) -> pd.DataFrame:
    """Run a BigQuery SQL query and return a pandas DataFrame."""
    print(f"\nQuerying {label}...")
    df = pd.read_gbq(
        sql,
        dialect="standard",
        use_bqstorage_api=use_bqstorage,
        progress_bar_type="tqdm_notebook",
    )
    print("Rows:", df.shape[0])
    if "person_id" in df.columns:
        print("Unique individuals:", df["person_id"].nunique())
    return df


def safe_filename(x) -> str:
    """Convert a string to a safe filename component."""
    x = str(x)
    x = re.sub(r"[^A-Za-z0-9_\-]+", "_", x)
    x = re.sub(r"_+", "_", x)
    return x.strip("_")


# SQL Queries

## Conditions

In [ ]:
def pull_all_conditions() -> pd.DataFrame:
    """Pull all condition records over time."""
    sql = f"""
    SELECT
        co.person_id,
        co.condition_occurrence_id,

        co.condition_concept_id,
        c_standard.concept_name    AS condition_concept_name,
        c_standard.vocabulary_id   AS standard_vocabulary,
        c_standard.concept_code    AS standard_concept_code,

        co.condition_start_date,
        co.condition_start_datetime,
        co.condition_end_date,
        co.condition_end_datetime,

        co.condition_type_concept_id,
        c_type.concept_name        AS condition_type,

        co.condition_status_concept_id,
        c_status.concept_name      AS condition_status,

        co.stop_reason,
        co.provider_id,
        co.visit_occurrence_id,
        co.visit_detail_id,

        co.condition_source_value,
        co.condition_source_concept_id,
        c_source.concept_name      AS source_concept_name,
        c_source.vocabulary_id     AS source_vocabulary,
        c_source.concept_code      AS source_concept_code,

        co.condition_status_source_value

    FROM `{cdr}.condition_occurrence` co

    LEFT JOIN `{cdr}.concept` c_standard
        ON co.condition_concept_id = c_standard.concept_id
    LEFT JOIN `{cdr}.concept` c_type
        ON co.condition_type_concept_id = c_type.concept_id
    LEFT JOIN `{cdr}.concept` c_status
        ON co.condition_status_concept_id = c_status.concept_id
    LEFT JOIN `{cdr}.concept` c_source
        ON co.condition_source_concept_id = c_source.concept_id

    ORDER BY
        co.person_id,
        COALESCE(co.condition_start_datetime, TIMESTAMP(co.condition_start_date)),
        co.condition_occurrence_id
    """
    return run_gbq_query(sql, "all condition records over time")



## Procedures

In [ ]:
def pull_all_procedures() -> pd.DataFrame:
    """Pull all procedure records over time."""
    sql = f"""
    SELECT
        po.person_id,
        po.procedure_occurrence_id,

        po.procedure_concept_id,
        c_standard.concept_name    AS procedure_concept_name,
        c_standard.vocabulary_id   AS standard_vocabulary,
        c_standard.concept_code    AS standard_concept_code,

        po.procedure_date,
        po.procedure_datetime,

        po.procedure_type_concept_id,
        c_type.concept_name        AS procedure_type,

        po.modifier_concept_id,
        c_modifier.concept_name    AS modifier_concept_name,

        po.quantity,
        po.provider_id,
        po.visit_occurrence_id,
        po.visit_detail_id,

        po.procedure_source_value,
        po.procedure_source_concept_id,
        c_source.concept_name      AS source_concept_name,
        c_source.vocabulary_id     AS source_vocabulary,
        c_source.concept_code      AS source_concept_code,

        po.modifier_source_value

    FROM `{cdr}.procedure_occurrence` po

    LEFT JOIN `{cdr}.concept` c_standard
        ON po.procedure_concept_id = c_standard.concept_id
    LEFT JOIN `{cdr}.concept` c_type
        ON po.procedure_type_concept_id = c_type.concept_id
    LEFT JOIN `{cdr}.concept` c_modifier
        ON po.modifier_concept_id = c_modifier.concept_id
    LEFT JOIN `{cdr}.concept` c_source
        ON po.procedure_source_concept_id = c_source.concept_id

    ORDER BY
        po.person_id,
        COALESCE(po.procedure_datetime, TIMESTAMP(po.procedure_date)),
        po.procedure_occurrence_id
    """
    return run_gbq_query(sql, "all procedure records over time")



## Drugs

In [ ]:
def pull_all_drugs() -> pd.DataFrame:
    """Pull all drug exposure records over time."""
    sql = f"""
    SELECT
        de.person_id,
        de.drug_exposure_id,

        de.drug_concept_id,
        c_standard.concept_name    AS drug_concept_name,
        c_standard.vocabulary_id   AS standard_vocabulary,
        c_standard.concept_code    AS standard_concept_code,

        de.drug_exposure_start_date,
        de.drug_exposure_start_datetime,
        de.drug_exposure_end_date,
        de.drug_exposure_end_datetime,
        de.verbatim_end_date,

        de.drug_type_concept_id,
        c_type.concept_name        AS drug_type,

        de.stop_reason,
        de.refills,
        de.quantity,
        de.days_supply,
        de.sig,

        de.route_concept_id,
        c_route.concept_name       AS route_concept_name,

        de.lot_number,
        de.provider_id,
        de.visit_occurrence_id,
        de.visit_detail_id,

        de.drug_source_value,
        de.drug_source_concept_id,
        c_source.concept_name      AS source_concept_name,
        c_source.vocabulary_id     AS source_vocabulary,
        c_source.concept_code      AS source_concept_code,

        de.route_source_value,
        de.dose_unit_source_value

    FROM `{cdr}.drug_exposure` de

    LEFT JOIN `{cdr}.concept` c_standard
        ON de.drug_concept_id = c_standard.concept_id
    LEFT JOIN `{cdr}.concept` c_type
        ON de.drug_type_concept_id = c_type.concept_id
    LEFT JOIN `{cdr}.concept` c_route
        ON de.route_concept_id = c_route.concept_id
    LEFT JOIN `{cdr}.concept` c_source
        ON de.drug_source_concept_id = c_source.concept_id

    ORDER BY
        de.person_id,
        COALESCE(de.drug_exposure_start_datetime, TIMESTAMP(de.drug_exposure_start_date)),
        de.drug_exposure_id
    """
    return run_gbq_query(sql, "all drug exposure records over time")



## Measurements

In [ ]:

def pull_all_measurements() -> pd.DataFrame:
    """Pull all measurement/lab records over time."""
    sql = f"""
    SELECT
        m.person_id,
        m.measurement_id,

        m.measurement_concept_id,
        c_standard.concept_name    AS measurement_concept_name,
        c_standard.vocabulary_id   AS standard_vocabulary,
        c_standard.concept_code    AS standard_concept_code,

        m.measurement_date,
        m.measurement_datetime,
        m.measurement_time,

        m.measurement_type_concept_id,
        c_type.concept_name        AS measurement_type,

        m.operator_concept_id,
        c_operator.concept_name    AS operator_concept_name,

        m.value_as_number,

        m.value_as_concept_id,
        c_value.concept_name       AS value_as_concept_name,

        m.unit_concept_id,
        c_unit.concept_name        AS unit_concept_name,

        m.range_low,
        m.range_high,

        m.provider_id,
        m.visit_occurrence_id,
        m.visit_detail_id,

        m.measurement_source_value,
        m.measurement_source_concept_id,
        c_source.concept_name      AS source_concept_name,
        c_source.vocabulary_id     AS source_vocabulary,
        c_source.concept_code      AS source_concept_code,

        m.unit_source_value,
        m.value_source_value

    FROM `{cdr}.measurement` m

    LEFT JOIN `{cdr}.concept` c_standard
        ON m.measurement_concept_id = c_standard.concept_id
    LEFT JOIN `{cdr}.concept` c_type
        ON m.measurement_type_concept_id = c_type.concept_id
    LEFT JOIN `{cdr}.concept` c_operator
        ON m.operator_concept_id = c_operator.concept_id
    LEFT JOIN `{cdr}.concept` c_value
        ON m.value_as_concept_id = c_value.concept_id
    LEFT JOIN `{cdr}.concept` c_unit
        ON m.unit_concept_id = c_unit.concept_id
    LEFT JOIN `{cdr}.concept` c_source
        ON m.measurement_source_concept_id = c_source.concept_id

    ORDER BY
        m.person_id,
        COALESCE(m.measurement_datetime, TIMESTAMP(m.measurement_date)),
        m.measurement_id
    """
    return run_gbq_query(sql, "all measurement/lab records over time")



## Observations

In [ ]:


def pull_all_observations() -> pd.DataFrame:
    """Pull all observation records over time."""
    sql = f"""
    SELECT
        o.person_id,
        o.observation_id,

        o.observation_concept_id,
        c_standard.concept_name    AS observation_concept_name,
        c_standard.vocabulary_id   AS standard_vocabulary,
        c_standard.concept_code    AS standard_concept_code,

        o.observation_date,
        o.observation_datetime,

        o.observation_type_concept_id,
        c_type.concept_name        AS observation_type,

        o.value_as_number,
        o.value_as_string,

        o.value_as_concept_id,
        c_value.concept_name       AS value_as_concept_name,

        o.qualifier_concept_id,
        c_qualifier.concept_name   AS qualifier_concept_name,

        o.unit_concept_id,
        c_unit.concept_name        AS unit_concept_name,

        o.provider_id,
        o.visit_occurrence_id,
        o.visit_detail_id,

        o.observation_source_value,
        o.observation_source_concept_id,
        c_source.concept_name      AS source_concept_name,
        c_source.vocabulary_id     AS source_vocabulary,
        c_source.concept_code      AS source_concept_code,

        o.unit_source_value,
        o.qualifier_source_value,
        o.value_source_value,

        o.observation_event_id,
        o.obs_event_field_concept_id

    FROM `{cdr}.observation` o

    LEFT JOIN `{cdr}.concept` c_standard
        ON o.observation_concept_id = c_standard.concept_id
    LEFT JOIN `{cdr}.concept` c_type
        ON o.observation_type_concept_id = c_type.concept_id
    LEFT JOIN `{cdr}.concept` c_value
        ON o.value_as_concept_id = c_value.concept_id
    LEFT JOIN `{cdr}.concept` c_qualifier
        ON o.qualifier_concept_id = c_qualifier.concept_id
    LEFT JOIN `{cdr}.concept` c_unit
        ON o.unit_concept_id = c_unit.concept_id
    LEFT JOIN `{cdr}.concept` c_source
        ON o.observation_source_concept_id = c_source.concept_id

    ORDER BY
        o.person_id,
        COALESCE(o.observation_datetime, TIMESTAMP(o.observation_date)),
        o.observation_id
    """
    return run_gbq_query(sql, "all observation records over time")


def pull_all_observations():
    sql = f"SELECT * FROM {cdr}.observation LIMIT 20"
    return run_gbq_query(sql, "all observation records")

## Visits

In [ ]:

def pull_all_visits() -> pd.DataFrame:
    """Pull all visit records over time."""
    sql = f"""
    SELECT
        vo.person_id,
        vo.visit_occurrence_id,

        vo.visit_concept_id,
        c_standard.concept_name    AS visit_concept_name,
        c_standard.vocabulary_id   AS standard_vocabulary,
        c_standard.concept_code    AS standard_concept_code,

        vo.visit_start_date,
        vo.visit_start_datetime,
        vo.visit_end_date,
        vo.visit_end_datetime,

        vo.visit_type_concept_id,
        c_type.concept_name        AS visit_type,

        vo.provider_id,
        vo.care_site_id,

        vo.visit_source_value,
        vo.visit_source_concept_id,
        c_source.concept_name      AS source_concept_name,
        c_source.vocabulary_id     AS source_vocabulary,
        c_source.concept_code      AS source_concept_code,

        vo.admitted_from_concept_id,
        c_admit.concept_name       AS admitted_from_concept_name,

        vo.admitted_from_source_value,

        vo.discharged_to_concept_id,
        c_discharge.concept_name   AS discharged_to_concept_name,

        vo.discharged_to_source_value,
        vo.preceding_visit_occurrence_id

    FROM `{cdr}.visit_occurrence` vo

    LEFT JOIN `{cdr}.concept` c_standard
        ON vo.visit_concept_id = c_standard.concept_id
    LEFT JOIN `{cdr}.concept` c_type
        ON vo.visit_type_concept_id = c_type.concept_id
    LEFT JOIN `{cdr}.concept` c_source
        ON vo.visit_source_concept_id = c_source.concept_id
    LEFT JOIN `{cdr}.concept` c_admit
        ON vo.admitted_from_concept_id = c_admit.concept_id
    LEFT JOIN `{cdr}.concept` c_discharge
        ON vo.discharged_to_concept_id = c_discharge.concept_id

    ORDER BY
        vo.person_id,
        COALESCE(vo.visit_start_datetime, TIMESTAMP(vo.visit_start_date)),
        vo.visit_occurrence_id
    """
    return run_gbq_query(sql, "all visit records over time")



## Devices

In [ ]:

def pull_all_devices() -> pd.DataFrame:
    """Pull all device exposure records over time."""
    sql = f"""
    SELECT
        de.person_id,
        de.device_exposure_id,

        de.device_concept_id,
        c_standard.concept_name    AS device_concept_name,
        c_standard.vocabulary_id   AS standard_vocabulary,
        c_standard.concept_code    AS standard_concept_code,

        de.device_exposure_start_date,
        de.device_exposure_start_datetime,
        de.device_exposure_end_date,
        de.device_exposure_end_datetime,

        de.device_type_concept_id,
        c_type.concept_name        AS device_type,

        de.unique_device_id,
        de.quantity,

        de.provider_id,
        de.visit_occurrence_id,
        de.visit_detail_id,

        de.device_source_value,
        de.device_source_concept_id,
        c_source.concept_name      AS source_concept_name,
        c_source.vocabulary_id     AS source_vocabulary,
        c_source.concept_code      AS source_concept_code

    FROM `{cdr}.device_exposure` de

    LEFT JOIN `{cdr}.concept` c_standard
        ON de.device_concept_id = c_standard.concept_id
    LEFT JOIN `{cdr}.concept` c_type
        ON de.device_type_concept_id = c_type.concept_id
    LEFT JOIN `{cdr}.concept` c_source
        ON de.device_source_concept_id = c_source.concept_id

    ORDER BY
        de.person_id,
        COALESCE(de.device_exposure_start_datetime, TIMESTAMP(de.device_exposure_start_date)),
        de.device_exposure_id
    """
    return run_gbq_query(sql, "all device exposure records over time")



## Surveys

In [ ]:
def pull_all_survey_conduct() -> pd.DataFrame:
    """Pull all survey conduct records."""
    sql = f"""
    SELECT
        sc.person_id,
        sc.survey_conduct_id,

        sc.survey_concept_id,
        c_survey.concept_name      AS survey_name,
        c_survey.vocabulary_id     AS survey_vocabulary,
        c_survey.concept_code      AS survey_concept_code,

        sc.survey_start_date,
        sc.survey_start_datetime,
        sc.survey_end_date,
        sc.survey_end_datetime,

        sc.provider_id,
        sc.assisted_concept_id,
        c_assisted.concept_name    AS assisted_concept_name,

        sc.respondent_type_concept_id,
        c_respondent.concept_name  AS respondent_type,

        sc.timing_concept_id,
        c_timing.concept_name      AS timing_concept_name,

        sc.collection_method_concept_id,
        c_collection.concept_name  AS collection_method,

        sc.survey_source_value,
        sc.survey_source_concept_id,
        c_source.concept_name      AS survey_source_concept_name,
        c_source.vocabulary_id     AS survey_source_vocabulary,
        c_source.concept_code      AS survey_source_concept_code,

        sc.survey_source_identifier,
        sc.validated_survey_concept_id,
        c_validated.concept_name   AS validated_survey_name

    FROM `{cdr}.survey_conduct` sc

    LEFT JOIN `{cdr}.concept` c_survey
        ON sc.survey_concept_id = c_survey.concept_id
    LEFT JOIN `{cdr}.concept` c_assisted
        ON sc.assisted_concept_id = c_assisted.concept_id
    LEFT JOIN `{cdr}.concept` c_respondent
        ON sc.respondent_type_concept_id = c_respondent.concept_id
    LEFT JOIN `{cdr}.concept` c_timing
        ON sc.timing_concept_id = c_timing.concept_id
    LEFT JOIN `{cdr}.concept` c_collection
        ON sc.collection_method_concept_id = c_collection.concept_id
    LEFT JOIN `{cdr}.concept` c_source
        ON sc.survey_source_concept_id = c_source.concept_id
    LEFT JOIN `{cdr}.concept` c_validated
        ON sc.validated_survey_concept_id = c_validated.concept_id

    ORDER BY
        sc.person_id,
        COALESCE(sc.survey_start_datetime, TIMESTAMP(sc.survey_start_date)),
        sc.survey_conduct_id
    """
    return run_gbq_query(sql, "all survey conduct records")


def pull_available_surveys() -> pd.DataFrame:
    """List all available surveys in survey_conduct."""
    sql = f"""
    SELECT
        sc.survey_concept_id,
        c_survey.concept_name  AS survey_name,
        c_survey.vocabulary_id AS survey_vocabulary,
        c_survey.concept_code  AS survey_concept_code,

        COUNT(*) AS n_survey_records,
        COUNT(DISTINCT sc.person_id) AS n_people,

        MIN(sc.survey_start_date) AS first_survey_date,
        MAX(sc.survey_start_date) AS last_survey_date

    FROM `{cdr}.survey_conduct` sc

    LEFT JOIN `{cdr}.concept` c_survey
        ON sc.survey_concept_id = c_survey.concept_id

    GROUP BY
        sc.survey_concept_id,
        survey_name,
        survey_vocabulary,
        survey_concept_code

    ORDER BY
        n_people DESC,
        survey_name
    """
    return run_gbq_query(sql, "available survey names")


def pull_all_survey_responses() -> pd.DataFrame:
    """Pull all question-answer level survey responses linked to survey_conduct."""
    sql = f"""
    SELECT
        o.person_id,
        o.observation_id,

        o.questionnaire_response_id,

        sc.survey_concept_id,
        c_survey.concept_name      AS survey_name,
        c_survey.vocabulary_id     AS survey_vocabulary,
        c_survey.concept_code      AS survey_concept_code,

        sc.survey_start_date,
        sc.survey_start_datetime,
        sc.survey_end_date,
        sc.survey_end_datetime,

        o.observation_concept_id,
        c_question.concept_name    AS question,
        c_question.vocabulary_id   AS question_vocabulary,
        c_question.concept_code    AS question_concept_code,

        o.observation_date,
        o.observation_datetime,
        o.observation_type_concept_id,
        c_type.concept_name        AS observation_type,

        o.value_as_number,
        o.value_as_string,

        o.value_as_concept_id,
        c_answer.concept_name      AS answer,
        c_answer.vocabulary_id     AS answer_vocabulary,
        c_answer.concept_code      AS answer_concept_code,

        o.qualifier_concept_id,
        c_qualifier.concept_name   AS qualifier,

        o.unit_concept_id,
        c_unit.concept_name        AS unit,

        o.provider_id,
        o.visit_occurrence_id,
        o.visit_detail_id,

        o.observation_source_value,
        o.observation_source_concept_id,
        c_source.concept_name      AS source_question_name,
        c_source.vocabulary_id     AS source_question_vocabulary,
        c_source.concept_code      AS source_question_code,

        o.unit_source_value,
        o.qualifier_source_value,
        o.value_source_value,

        o.observation_event_id,
        o.obs_event_field_concept_id

    FROM `{cdr}.observation` o

    LEFT JOIN `{cdr}.concept` c_question
        ON o.observation_concept_id = c_question.concept_id
    LEFT JOIN `{cdr}.concept` c_type
        ON o.observation_type_concept_id = c_type.concept_id
    LEFT JOIN `{cdr}.concept` c_answer
        ON o.value_as_concept_id = c_answer.concept_id
    LEFT JOIN `{cdr}.concept` c_qualifier
        ON o.qualifier_concept_id = c_qualifier.concept_id
    LEFT JOIN `{cdr}.concept` c_unit
        ON o.unit_concept_id = c_unit.concept_id
    LEFT JOIN `{cdr}.concept` c_source
        ON o.observation_source_concept_id = c_source.concept_id

    WHERE o.survey_conduct_id IS NOT NULL AND o.observation_concept_id IS NOT NULL

    ORDER BY
        o.person_id,
        COALESCE(sc.survey_start_datetime, TIMESTAMP(sc.survey_start_date),
                 o.observation_datetime, TIMESTAMP(o.observation_date)),
        sc.survey_conduct_id,
        o.observation_id
    """
    return run_gbq_query(sql, "all survey question-answer responses")


def pull_survey_observations_fallback() -> pd.DataFrame:
    """
    Fallback survey-like observation pull for CDR versions where observation.survey_conduct_id
    is not available or not populated.
    """
    sql = f"""
    SELECT
        o.person_id,
        o.observation_id,

        o.observation_concept_id,
        c_question.concept_name    AS question,
        c_question.vocabulary_id   AS question_vocabulary,
        c_question.concept_code    AS question_concept_code,

        o.observation_date,
        o.observation_datetime,

        o.observation_type_concept_id,
        c_type.concept_name        AS observation_type,

        o.value_as_number,
        o.value_as_string,

        o.value_as_concept_id,
        c_answer.concept_name      AS answer,
        c_answer.vocabulary_id     AS answer_vocabulary,
        c_answer.concept_code      AS answer_concept_code,

        o.qualifier_concept_id,
        c_qualifier.concept_name   AS qualifier,

        o.unit_concept_id,
        c_unit.concept_name        AS unit,

        o.provider_id,
        o.visit_occurrence_id,
        o.visit_detail_id,

        o.observation_source_value,
        o.observation_source_concept_id,
        c_source.concept_name      AS source_question_name,
        c_source.vocabulary_id     AS source_question_vocabulary,
        c_source.concept_code      AS source_question_code,

        o.unit_source_value,
        o.qualifier_source_value,
        o.value_source_value,


    FROM `{cdr}.observation` o

    LEFT JOIN `{cdr}.concept` c_question
        ON o.observation_concept_id = c_question.concept_id
    LEFT JOIN `{cdr}.concept` c_type
        ON o.observation_type_concept_id = c_type.concept_id
    LEFT JOIN `{cdr}.concept` c_answer
        ON o.value_as_concept_id = c_answer.concept_id
    LEFT JOIN `{cdr}.concept` c_qualifier
        ON o.qualifier_concept_id = c_qualifier.concept_id
    LEFT JOIN `{cdr}.concept` c_unit
        ON o.unit_concept_id = c_unit.concept_id
    LEFT JOIN `{cdr}.concept` c_source
        ON o.observation_source_concept_id = c_source.concept_id

    WHERE
        LOWER(c_question.vocabulary_id) LIKE '%ppi%'
        OR LOWER(c_source.vocabulary_id) LIKE '%ppi%'
        OR LOWER(c_question.concept_name) LIKE '%survey%'
        OR LOWER(c_source.concept_name) LIKE '%survey%' 

    ORDER BY
        o.person_id,
        COALESCE(o.observation_datetime, TIMESTAMP(o.observation_date)),
        o.observation_id
    """
    return run_gbq_query(sql, "survey-like observations fallback")

    
def pull_survey_timeline() -> pd.DataFrame:
    """Pull an LLM-ready survey response timeline table."""
    sql = f"""
    SELECT
        o.person_id,

        COALESCE(
            sc.survey_start_datetime,
            TIMESTAMP(sc.survey_start_date),
            o.observation_datetime,
            TIMESTAMP(o.observation_date)
        ) AS event_datetime,

        DATE(COALESCE(
            sc.survey_start_datetime,
            TIMESTAMP(sc.survey_start_date),
            o.observation_datetime,
            TIMESTAMP(o.observation_date)
        )) AS event_date,

        'survey' AS domain,

        o.questionnaire_response_id,
        sc.survey_conduct_id,
        sc.survey_concept_id,
        c_survey.concept_name AS survey_name,

        o.observation_id,
        o.observation_concept_id,
        c_question.concept_name AS question,

        o.value_as_number,
        o.value_as_string,
        o.value_as_concept_id,
        c_answer.concept_name AS answer,

        o.unit_concept_id,
        c_unit.concept_name AS unit,

        o.observation_source_value,
        o.value_source_value,

        CONCAT(
            'Survey: ', COALESCE(c_survey.concept_name, 'Unknown survey'),
            ' | Question: ', COALESCE(c_question.concept_name, o.observation_source_value, 'Unknown question'),
            ' | Answer: ',
                COALESCE(
                    c_answer.concept_name,
                    CAST(o.value_as_number AS STRING),
                    o.value_as_string,
                    o.value_source_value,
                    'Unknown answer'
                ),
            CASE
                WHEN c_unit.concept_name IS NOT NULL
                THEN CONCAT(' ', c_unit.concept_name)
                ELSE ''
            END,
            ' | Time and Date: ', COALESCE(
            sc.survey_start_datetime,
            TIMESTAMP(sc.survey_start_date),
            o.observation_datetime,
            TIMESTAMP(o.observation_date)
        )
            
        ) AS llm_event_text

    FROM `{cdr}.observation` o

    LEFT JOIN `{cdr}.survey_conduct` sc
        ON o.questionnaire_response_id = sc.survey_conduct_id

    LEFT JOIN `{cdr}.concept` c_survey
        ON sc.survey_concept_id = c_survey.concept_id

    LEFT JOIN `{cdr}.concept` c_question
        ON o.observation_concept_id = c_question.concept_id

    LEFT JOIN `{cdr}.concept` c_answer
        ON o.value_as_concept_id = c_answer.concept_id

    LEFT JOIN `{cdr}.concept` c_unit
        ON o.unit_concept_id = c_unit.concept_id

    WHERE o.questionnaire_response_id IS NOT NULL

    ORDER BY
        person_id,
        event_datetime,
        questionnaire_response_id,
        observation_id
    """
    return run_gbq_query(sql, "LLM-ready survey response timeline")


## Individual Survey

In [ ]:

def pull_one_survey_responses(survey_concept_id: int) -> pd.DataFrame:
    """Pull responses for one specific survey_concept_id."""
    sql = f"""
    SELECT
        o.person_id,
        o.observation_id,

        o.questionnaire_response_id,
        o.survey_conduct_id,

        sc.survey_concept_id,
        c_survey.concept_name      AS survey_name,
        c_survey.vocabulary_id     AS survey_vocabulary,
        c_survey.concept_code      AS survey_concept_code,

        sc.survey_start_date,
        sc.survey_start_datetime,
        sc.survey_end_date,
        sc.survey_end_datetime,

        o.observation_concept_id,
        c_question.concept_name    AS question,
        c_question.vocabulary_id   AS question_vocabulary,
        c_question.concept_code    AS question_concept_code,

        o.observation_date,
        o.observation_datetime,

        o.value_as_number,
        o.value_as_string,

        o.value_as_concept_id,
        c_answer.concept_name      AS answer,
        c_answer.vocabulary_id     AS answer_vocabulary,
        c_answer.concept_code      AS answer_concept_code,

        o.qualifier_concept_id,
        c_qualifier.concept_name   AS qualifier,

        o.unit_concept_id,
        c_unit.concept_name        AS unit,

        o.observation_source_value,
        o.observation_source_concept_id,
        c_source.concept_name      AS source_question_name,
        c_source.vocabulary_id     AS source_question_vocabulary,
        c_source.concept_code      AS source_question_code,

        o.unit_source_value,
        o.qualifier_source_value,
        o.value_source_value

    FROM `{cdr}.observation` o

    INNER JOIN `{cdr}.survey_conduct` sc
        ON o.survey_conduct_id = sc.survey_conduct_id
    LEFT JOIN `{cdr}.concept` c_survey
        ON sc.survey_concept_id = c_survey.concept_id
    LEFT JOIN `{cdr}.concept` c_question
        ON o.observation_concept_id = c_question.concept_id
    LEFT JOIN `{cdr}.concept` c_answer
        ON o.value_as_concept_id = c_answer.concept_id
    LEFT JOIN `{cdr}.concept` c_qualifier
        ON o.qualifier_concept_id = c_qualifier.concept_id
    LEFT JOIN `{cdr}.concept` c_unit
        ON o.unit_concept_id = c_unit.concept_id
    LEFT JOIN `{cdr}.concept` c_source
        ON o.observation_source_concept_id = c_source.concept_id

    WHERE sc.survey_concept_id = {survey_concept_id}

    ORDER BY
        o.person_id,
        COALESCE(sc.survey_start_datetime, TIMESTAMP(sc.survey_start_date),
                 o.observation_datetime, TIMESTAMP(o.observation_date)),
        o.observation_id
    """
    return run_gbq_query(sql, f"responses for survey_concept_id={survey_concept_id}")


# Save helpers

In [ ]:

def save_df_to_bucket(df: pd.DataFrame, folder_name:str, filename: str, compression: str = "gzip") -> str:
    """Save a DataFrame to the workspace bucket under /{folder_name}."""
    if bucket is None:
        raise EnvironmentError("WORKSPACE_BUCKET is not set. Cannot save to bucket.")

    if compression == "gzip" and not filename.endswith(".gz"):
        filename = filename + ".gz"

    out_path = f"{bucket}/{folder_name}/{filename}"
    df.to_csv(out_path, index=False, compression=compression)
    print("Saved to:", out_path)
    return out_path


def pull_and_save_core_ehr_tables() -> dict:
    """
    Pull all core longitudinal EHR/event tables and save them as compressed CSV files.
    Warning: These tables can be very large.
    """
    outputs = {}

    all_conditions_df = pull_all_conditions()
    outputs["conditions"] = save_df_to_bucket(
        all_conditions_df, "all_conditions_longitudinal.csv.gz"
    )

    all_procedures_df = pull_all_procedures()
    outputs["procedures"] = save_df_to_bucket(
        all_procedures_df, "all_procedures_longitudinal.csv.gz"
    )

    all_drugs_df = pull_all_drugs()
    outputs["drugs"] = save_df_to_bucket(
        all_drugs_df, "all_drugs_longitudinal.csv.gz"
    )

    all_measurements_df = pull_all_measurements()
    outputs["measurements"] = save_df_to_bucket(
        all_measurements_df, "all_measurements_longitudinal.csv.gz"
    )

    all_observations_df = pull_all_observations()
    outputs["observations"] = save_df_to_bucket(
        all_observations_df, "all_observations_longitudinal.csv.gz"
    )

    all_visits_df = pull_all_visits()
    outputs["visits"] = save_df_to_bucket(
        all_visits_df, "all_visits_longitudinal.csv.gz"
    )

    all_devices_df = pull_all_devices()
    outputs["devices"] = save_df_to_bucket(
        all_devices_df, "all_devices_longitudinal.csv.gz"
    )

    return outputs


def pull_and_save_survey_tables() -> dict:
    """Pull survey conduct, survey responses, and survey LLM timeline and save them."""
    outputs = {}

    all_survey_conduct_df = pull_all_survey_conduct()
    outputs["survey_conduct"] = save_df_to_bucket(
        all_survey_conduct_df, "all_survey_conduct.csv.gz"
    )

    available_surveys_df = pull_available_surveys()
    outputs["available_surveys"] = save_df_to_bucket(
        available_surveys_df, "available_surveys.csv.gz"
    )

    all_survey_responses_df = pull_all_survey_responses()
    outputs["survey_responses"] = save_df_to_bucket(
        all_survey_responses_df, "all_survey_responses.csv.gz"
    )

    survey_timeline_df = pull_survey_timeline()
    outputs["survey_timeline"] = save_df_to_bucket(
        survey_timeline_df, "all_survey_responses_llm_timeline.csv.gz"
    )

    return outputs


def pull_each_survey_and_save() -> dict:
    """Pull each survey separately by survey_concept_id and save as compressed CSV."""
    available_surveys_df = pull_available_surveys()
    outputs = {}

    for _, row in available_surveys_df.iterrows():
        survey_concept_id = int(row["survey_concept_id"])
        survey_name = row["survey_name"]
        survey_label = safe_filename(survey_name)

        df = pull_one_survey_responses(survey_concept_id)

        filename = f"survey_{survey_concept_id}_{survey_label}.csv.gz"
        outputs[survey_concept_id] = save_df_to_bucket(df, filename)

    return outputs


# Memory-safe export

In [ ]:

def export_query_to_gcs(sql: str, destination_uri: str, temp_table_name: str) -> str:
    """
    Run a SQL query into a temporary BigQuery table, then export the result to GCS.

    This is useful for very large tables that should not be loaded into pandas.

    Parameters
    ----------
    sql : str
        BigQuery SQL query.
    destination_uri : str
        GCS URI, for example:
        gs://my-bucket/data/my_export_*.csv.gz
    temp_table_name : str
        Fully qualified BigQuery destination table name, for example:
        my-project.scratch.my_temp_table
    """
    if bigquery is None:
        raise ImportError("google-cloud-bigquery is not installed.")

    client = bigquery.Client()

    query_job_config = bigquery.QueryJobConfig(
        destination=temp_table_name,
        write_disposition="WRITE_TRUNCATE",
    )

    print("Creating temporary BigQuery table:", temp_table_name)
    query_job = client.query(sql, job_config=query_job_config)
    query_job.result()

    extract_job_config = bigquery.ExtractJobConfig(
        destination_format=bigquery.DestinationFormat.CSV,
        compression=bigquery.Compression.GZIP,
        print_header=True,
    )

    print("Exporting to:", destination_uri)
    extract_job = client.extract_table(
        temp_table_name,
        destination_uri,
        job_config=extract_job_config,
    )
    extract_job.result()

    print("Export complete:", destination_uri)
    return destination_uri



# Pull Data

## Take tables from CDR and add them to Workspace Bucket in Parquet format

In [ ]:
import os
from google.cloud import bigquery

client = bigquery.Client(project=os.environ["GOOGLE_PROJECT"])
cdr = os.environ["WORKSPACE_CDR"]

tables = list(client.list_tables(cdr))

for table in tables:
    print(table.table_id)

In [ ]:
# ============================================================
# Export All of Us raw OMOP tables to GCS as compressed Parquet
# for OMOP-MEDS input.
#
# Output layout:
# gs://YOUR_BUCKET/data_pre_meds_raw/person/part-*.parquet
# gs://YOUR_BUCKET/data_pre_meds_raw/condition_occurrence/part-*.parquet
# ...
#
# This avoids pandas and avoids local Jupyter disk limits.
# ============================================================

import os
from google.cloud import bigquery

# -----------------------------
# Required All of Us env values
# -----------------------------
PROJECT = os.environ["GOOGLE_PROJECT"]
CDR = os.environ["WORKSPACE_CDR"]

# -----------------------------
# Manually set your NEW working bucket
# Do NOT use os.environ["WORKSPACE_BUCKET"] if it points to the broken old bucket.
# -----------------------------
DATA_BUCKET = 'gs://cloned-data-bucket-wb-jaunty-jalapeno-1964'
STAGING_DIR = "data_pre_meds_raw"
GCS_STAGING_DIR = f"{DATA_BUCKET}/{STAGING_DIR}"

print("PROJECT:", PROJECT)
print("CDR:", CDR)
print("Using bucket:", DATA_BUCKET)
print("GCS staging dir:", GCS_STAGING_DIR)

client = bigquery.Client(project=PROJECT)

# -----------------------------
# Tables relevant for OMOP-MEDS
# -----------------------------
OMOP_TABLES = [
    # Core person/static info
    "person",
    "observation_period",
    "death",

    # Health System
    'care_site',
    'location',
    'provider',

    # Core longitudinal EHR/event tables
    "condition_occurrence",
    "procedure_occurrence",
    "drug_exposure",
    "measurement",
    "observation",
    "visit_occurrence",
    "device_exposure",

    # Optional but often useful OMOP tables
    "visit_detail",
    "specimen",
    'note_nlp',
    'note'

    # Survey support
    "survey_conduct",

    # Vocabulary/metadata tables
    "concept",
    "domain",
    'vocabulary',
    'concept_ancestor',
    'concept_relationship',
    "relationship",
]

# -----------------------------
# Debug switch
# Keep this small first.
# Set LIMIT_ROWS = None only after the test works.
# -----------------------------
LIMIT_ROWS = None  # use None for full export


def gcloud_rm_prefix(prefix_uri: str):
    """
    Remove old files under a GCS prefix using gcloud storage.
    Safe if the prefix does not exist.
    """
    print(f"Cleaning old files under: {prefix_uri}")
    # The || true prevents the notebook from failing if the folder is empty/missing.
    !gcloud storage rm --recursive "{prefix_uri}/**" || true


def export_omop_table_to_gcs(
    table_name: str,
    limit_rows: int | None = None,
    overwrite: bool = True,
) -> str:
    """
    Export one raw OMOP table from the All of Us CDR to GCS as Parquet.
    Uses BigQuery EXPORT DATA directly, so no scratch dataset/table is needed.
    """
    table_uri = f"`{CDR}.{table_name}`"
    output_prefix = f"{GCS_STAGING_DIR}/{table_name}"
    destination_uri = f"{output_prefix}/part-*.parquet"

    if overwrite:
        gcloud_rm_prefix(output_prefix)

    sql = f"""
    SELECT *
    FROM {table_uri}
    """

    if limit_rows is not None:
        sql += f"\nLIMIT {int(limit_rows)}"

    export_sql = f"""
    EXPORT DATA OPTIONS(
      uri='{destination_uri}',
      format='PARQUET',
      compression='SNAPPY',
      overwrite=true
    ) AS
    {sql}
    """

    print("=" * 100)
    print(f"Exporting table: {table_name}")
    print(f"Destination:     {destination_uri}")
    print(f"LIMIT_ROWS:      {limit_rows}")

    job = client.query(export_sql)
    job.result()

    print(f"Finished: {table_name}")
    return destination_uri


# -----------------------------
# Test bucket write access first
# -----------------------------
print("\nTesting bucket access...")
test_uri = f"{DATA_BUCKET}/_test_write_from_notebook.txt"
!echo "test" | gcloud storage cp - "{test_uri}"
!gcloud storage ls "{test_uri}"
!gcloud storage rm "{test_uri}"
print("Bucket write test passed.")


# -----------------------------
# Export all selected tables
# -----------------------------
outputs = {}

for table_name in OMOP_TABLES:
    try:
        outputs[table_name] = export_omop_table_to_gcs(
            table_name=table_name,
            limit_rows=LIMIT_ROWS,
            overwrite=True,
        )
    except Exception as e:
        print("=" * 100)
        print(f"[ERROR] Failed to export {table_name}")
        print(type(e).__name__, str(e))
        outputs[table_name] = None


# -----------------------------
# Summary
# -----------------------------
print("\n" + "=" * 100)
print("EXPORT SUMMARY")
print("=" * 100)

for table_name, uri in outputs.items():
    status = "OK" if uri is not None else "FAILED"
    print(f"{table_name:30s} {status:8s} {uri}")

print("\nStaged OMOP-MEDS raw_input_dir:")
print(GCS_STAGING_DIR)

print("\nListing staged files:")
!gcloud storage ls --recursive "{GCS_STAGING_DIR}/**"

## Generate MEDS data using OMOP-MEDS|

In [ ]:
!conda create -y --name omop_meds_py311 python=3.11

In [ ]:
!bash -lc "source activate omop_meds_py311 && python -m pip install --upgrade pip"
!bash -lc "source activate omop_meds_py311 && python -m pip install ipykernel"
!bash -lc "source activate omop_meds_py311 && python -m pip install OMOP-MEDS"

In [ ]:
!bash -lc "source /opt/conda/etc/profile.d/conda.sh && conda activate omop_meds_py311 && python -m ipykernel install --user --name omop_meds_py311 --display-name 'Python 3.11 OMOP-MEDS'"

In [ ]:
!bash -lc "source /opt/conda/etc/profile.d/conda.sh && conda activate omop_meds_py311 && OMOP_MEDS --help"

### SWITCH TO OMOP-MEDS ENVIRONMENT KERNEL!!!!

### Change OMOP-MEDS config to allow for lock file to not exist

In [ ]:
from pathlib import Path
import MEDS_transforms.mapreduce.utils as u

fp = Path(u.__file__)
print('Patching:', fp)

text = fp.read_text()

old = 'lock_fp.unlink()'
new = 'lock_fp.unlink(missing_ok=True)'

if new in text:
    print('Already patched.')
elif old in text:
    backup = fp.with_suffix(fp.suffix + '.bak')
    backup.write_text(text)
    fp.write_text(text.replace(old, new))
    print('Patched successfully.')
    print('Backup saved to:', backup)
else:
    print('Could not find exact lock_fp.unlink() line. Inspect manually.')

In [ ]:
from pathlib import Path
import MEDS_transforms.mapreduce.utils as u

fp = Path(u.__file__)
print('Patching:', fp)

text = fp.read_text()

old = 'lock_fp.unlink()'
new = 'lock_fp.unlink(missing_ok=True)'

if new in text:
    print('Already patched.')
elif old in text:
    backup = fp.with_suffix(fp.suffix + '.bak')
    backup.write_text(text)
    fp.write_text(text.replace(old, new))
    print('Patched successfully.')
    print('Backup saved to:', backup)
else:
    print('Could not find exact lock_fp.unlink() line. Inspect manually.')

fp = Path(u.__file__)
for i, line in enumerate(fp.read_text().splitlines(), start=1):
    if 'lock_fp.unlink' in line:
        print(i, line)

In [ ]:
import MEDS_transforms.mapreduce.utils as u
from pathlib import Path

fp = Path(u.__file__)
for i, line in enumerate(fp.read_text().splitlines(), start=1):
    if 'lock_fp.unlink' in line:
        print(i, line)


### Patch OMOP-MEDS to read concept_ID columns

In [ ]:
from pathlib import Path
import shutil
import yaml

CONFIG_PATH = Path(
    "/opt/conda/envs/omop_meds_py311/lib/python3.11/site-packages/"
    "OMOP_MEDS/configs/pre_MEDS_minimal.yaml"
)

BACKUP_PATH = CONFIG_PATH.with_suffix(".yaml.bak")

# Backup original file once
if not BACKUP_PATH.exists():
    shutil.copy2(CONFIG_PATH, BACKUP_PATH)
    print(f"Backup written to: {BACKUP_PATH}")
else:
    print(f"Backup already exists: {BACKUP_PATH}")

with open(CONFIG_PATH, "r") as f:
    cfg = yaml.safe_load(f)

def ensure_cols(table, version, cols_to_add):
    """
    Add cols_to_add to cfg[table][version]["output_data_cols"]
    if they are not already present.
    """
    block = cfg[table][version]
    out_cols = list(block.get("output_data_cols", []))

    for col in cols_to_add:
        if col not in out_cols:
            out_cols.insert(0, col)

    block["output_data_cols"] = out_cols

# OMOP 5.3: these were in reference_cols but not kept in output_data_cols
ensure_cols(
    table="observation",
    version=5.3,
    cols_to_add=[
        "observation_id",
        "observation_concept_id",
        "observation_date",
    ],
)

ensure_cols(
    table="measurement",
    version=5.3,
    cols_to_add=[
        "measurement_id",
        "measurement_concept_id",
        "measurement_date",
        "measurement_type_concept_id",
        "measurement_source_concept_id",
    ],
)

# Visit concept ID was also not present in your pre_MEDS visit table.
# This should work if the raw OMOP visit_occurrence table has visit_concept_id,
# which standard OMOP does.
ensure_cols(
    table="visit_occurrence",
    version=5.3,
    cols_to_add=[
        "visit_concept_id",
        "visit_source_concept_id",
    ],
)

with open(CONFIG_PATH, "w") as f:
    yaml.safe_dump(cfg, f, sort_keys=False)

print(f"Updated config written to: {CONFIG_PATH}")

In [ ]:
from pathlib import Path
import shutil

EVENT_CONFIG = Path(
    "/opt/conda/envs/omop_meds_py311/lib/python3.11/site-packages/"
    "OMOP_MEDS/configs/event_configs.yaml"
)

backup = EVENT_CONFIG.with_suffix(".yaml.bak")
if not backup.exists():
    shutil.copy2(EVENT_CONFIG, backup)
    print(f"Backup created: {backup}")
else:
    print(f"Backup already exists: {backup}")

event_yaml = r"""
subject_id_col: person_id

person_birth_death:
  death_time:
    code: MEDS_DEATH
    time: col(date_of_death)
    time_format: '%Y-%m-%d %H:%M:%S'
    table_name: table_name

  age:
    code:
      - MEDS_BIRTH
    time: col(date_of_birth)
    time_format: '%Y-%m-%d %H:%M:%S'
    table_name: table_name

person:
  race:
    code:
      - RACE
      - col(race_concept_id)
    time: null
    table_name: table_name

  gender:
    code:
      - GENDER
      - col(gender_source_concept_id)
    time: null
    table_name: table_name

  ethnicity:
    code:
      - ETHNICITY
      - col(ethnicity_concept_id)
    time: null
    table_name: table_name

observation:
  observation:
    code:
      - col(preferred_vocabulary_name)
      - col(concept_code)
    time: col(observation_datetime)
    numeric_value: col(value_as_number)
    unit: col(unit_concept_id)
    text_value: col(value_as_string)
    visit_occurrence_id: col(visit_occurrence_id)
    table_name: col(table_name)

measurement:
  measurement:
    code:
      - col(preferred_vocabulary_name)
      - col(concept_code)
    time: col(measurement_datetime)
    numeric_value: col(value_as_number)
    unit: col(unit_concept_id)
    visit_occurrence_id: col(visit_occurrence_id)
    table_name: col(table_name)

visit_occurrence:
  visit:
    code:
      - col(preferred_vocabulary_name)
      - col(concept_code)
      - start
    time: col(visit_start_datetime)
    visit_occurrence_id: col(visit_occurrence_id)
    text: col(care_site_name)
    table_name: col(table_name)

  visit_end:
    code:
      - col(preferred_vocabulary_name)
      - col(concept_code)
      - end
    time: col(visit_end_datetime)
    visit_occurrence_id: col(visit_occurrence_id)
    table_name: col(table_name)

observation_period:
  observation_period:
    code:
      - OBSERVATION_PERIOD
      - col(period_type_concept_id)
      - start
    time: col(observation_period_start_date)
    table_name: col(table_name)

  observation_period_end:
    code:
      - OBSERVATION_PERIOD
      - col(period_type_concept_id)
      - end
    time: col(observation_period_end_date)
    table_name: col(table_name)

drug_exposure:
  drug_exposure:
    code:
      - col(preferred_vocabulary_name)
      - col(concept_code)
      - start
    time: col(drug_exposure_start_datetime)
    numeric_value: col(quantity)
    visit_occurrence_id: col(visit_occurrence_id)
    table_name: col(table_name)

  drug_exposure_end:
    code:
      - col(preferred_vocabulary_name)
      - col(concept_code)
      - end
    time: col(drug_exposure_end_datetime)
    visit_occurrence_id: col(visit_occurrence_id)
    table_name: col(table_name)

specimen:
  specimen:
    code:
      - col(preferred_vocabulary_name)
      - col(concept_code)
    time: col(specimen_datetime)
    numeric_value: col(quantity)
    unit: col(unit_concept_id)
    table_name: col(table_name)

device_exposure:
  device_exposure:
    code:
      - col(preferred_vocabulary_name)
      - col(concept_code)
      - start
    time: col(device_exposure_start_datetime)
    numeric_value: col(quantity)
    visit_occurrence_id: col(visit_occurrence_id)
    table_name: col(table_name)

  device_exposure_end:
    code:
      - col(preferred_vocabulary_name)
      - col(concept_code)
      - end
    time: col(device_exposure_end_datetime)
    numeric_value: col(quantity)
    visit_occurrence_id: col(visit_occurrence_id)
    table_name: col(table_name)

condition_occurrence:
  condition_occurrence:
    code:
      - col(preferred_vocabulary_name)
      - col(concept_code)
      - start
    time: col(condition_start_datetime)
    visit_occurrence_id: col(visit_occurrence_id)
    table_name: col(table_name)

  condition_end:
    code:
      - col(preferred_vocabulary_name)
      - col(concept_code)
      - end
    time: col(condition_end_datetime)
    visit_occurrence_id: col(visit_occurrence_id)
    table_name: col(table_name)

procedure_occurrence:
  procedure_occurrence:
    code:
      - col(preferred_vocabulary_name)
      - col(concept_code)
    time: col(procedure_datetime)
    numeric_value: col(quantity)
    visit_occurrence_id: col(visit_occurrence_id)
    table_name: col(table_name)
"""

EVENT_CONFIG.write_text(event_yaml.strip() + "\n")
print(f"Updated: {EVENT_CONFIG}")

### Run OMOP-MEDS

In [ ]:
!pip install hydra-joblib-launcher --upgrade

In [ ]:
!export N_WORKERS=8

import os
import re

try:
    from google.cloud import bigquery
except ImportError:
    bigquery = None

MOUNTED_BUCKET = '/home/jupyter/workspace/data_bucket'
STAGING_DIR = "data_pre_meds_raw"
GCS_STAGING_DIR = f"{MOUNTED_BUCKET}/{STAGING_DIR}/"

YAML_CONFIG = '/home/jupyter/workspace/data_bucket/MEDS_DATA/event_configs.yaml'

MEDS_DIR = "MEDS_DATA"
GCS_MEDS_DIR = f"{MOUNTED_BUCKET}/{MEDS_DIR}/"

PRE_MEDS_DIR = f'{MOUNTED_BUCKET}/{MEDS_DIR}/pre_MEDS'

!OMOP_MEDS \
    root_output_dir={GCS_MEDS_DIR} \
    raw_input_dir={GCS_STAGING_DIR} \
    do_download=False \
    do_overwrite=False \
    pre_meds_batch_size_shards=10 \
    pre_meds_batching_row_threshold=1000000000 \
    join_on_visit=True \

print('Done generating MEDS data')

In [ ]:
!pip uninstall -y meds_etl
!pip install "git+https://github.com/Medical-Event-Data-Standard/meds_etl.git"

In [ ]:
!meds_etl_omop --help

In [8]:
# !pip install meds_etl
!pip install "meds_etl[cpp]"

In [ ]:
#!meds_etl_omop /home/jupyter/workspace/data_bucket/data_pre_meds_raw /home/jupyter/workspace/data_bucket/meds --omop_version 5.3 --num_proc 1

In [10]:
# !rm -rf /home/jupyter/meds_sorted
# !meds_etl_sort \
#     /home/jupyter/workspace/data_bucket/meds/temp \
#     /home/jupyter/meds_sorted \
#     --num_shards 100 \
#     --num_proc 1 \
#     --backend cpp

In [11]:
!rm -rf /home/jupyter/meds_sorted

from meds_etl.unsorted import sort

sort(
    source_unsorted_path="/home/jupyter/workspace/data_bucket/meds/temp",
    target_meds_path="/home/jupyter/meds_sorted",
    num_shards=100,
    num_proc=32,
    backend="cpp",
)

In [12]:
!rm -rf /home/jupyter/workspace/data_bucket/meds

In [13]:
!mv /home/jupyter/meds_sorted /home/jupyter/workspace/data_bucket/meds

In [ ]:
!pip install MEDS-Inspect
!MEDS_Inspect_cache "/home/jupyter/workspace/data_bucket/MEDS_DATA/MEDS_cohort"
!MEDS_Inspect port=8052 +initial_path="/home/jupyter/workspace/data_bucket/MEDS_DATA/MEDS_cohort"

## Load Survey Data

In [ ]:
client = bigquery.Client(project=os.environ["GOOGLE_PROJECT"])

DATA_BUCKET = 'gs://cloned-data-bucket-wb-jaunty-jalapeno-1964'
SURVEY_DIR = "survey_data"
GCS_DIR = f"{DATA_BUCKET}/{SURVEY_DIR}/part-*.parquet"

sql = f"""
SELECT
    o.person_id,

    COALESCE(
        sc.survey_start_datetime,
        TIMESTAMP(sc.survey_start_date),
        o.observation_datetime,
        TIMESTAMP(o.observation_date)
    ) AS event_datetime,

    DATE(COALESCE(
        sc.survey_start_datetime,
        TIMESTAMP(sc.survey_start_date),
        o.observation_datetime,
        TIMESTAMP(o.observation_date)
    )) AS event_date,

    'survey' AS domain,

    o.questionnaire_response_id,
    sc.survey_conduct_id,
    sc.survey_concept_id,
    c_survey.concept_name AS survey_name,

    o.observation_id,
    o.observation_concept_id,
    c_question.concept_name AS question,

    o.value_as_number,
    o.value_as_string,
    o.value_as_concept_id,
    c_answer.concept_name AS answer,

    o.unit_concept_id,
    c_unit.concept_name AS unit,

    o.observation_source_value,
    o.value_source_value,

    CONCAT(
        'Survey: ', COALESCE(c_survey.concept_name, 'Unknown survey'),
        ' | Question: ', COALESCE(c_question.concept_name, o.observation_source_value, 'Unknown question'),
        ' | Answer: ',
            COALESCE(
                c_answer.concept_name,
                CAST(o.value_as_number AS STRING),
                o.value_as_string,
                o.value_source_value,
                'Unknown answer'
            ),
        CASE
            WHEN c_unit.concept_name IS NOT NULL
            THEN CONCAT(' ', c_unit.concept_name)
            ELSE ''
        END,
        ' | Time and Date: ', COALESCE(
        sc.survey_start_datetime,
        TIMESTAMP(sc.survey_start_date),
        o.observation_datetime,
        TIMESTAMP(o.observation_date)
    )
        
    ) AS llm_event_text

FROM `{cdr}.observation` o

LEFT JOIN `{cdr}.survey_conduct` sc
    ON o.questionnaire_response_id = sc.survey_conduct_id

LEFT JOIN `{cdr}.concept` c_survey
    ON sc.survey_concept_id = c_survey.concept_id

LEFT JOIN `{cdr}.concept` c_question
    ON o.observation_concept_id = c_question.concept_id

LEFT JOIN `{cdr}.concept` c_answer
    ON o.value_as_concept_id = c_answer.concept_id

LEFT JOIN `{cdr}.concept` c_unit
    ON o.unit_concept_id = c_unit.concept_id

WHERE o.questionnaire_response_id IS NOT NULL

ORDER BY
    person_id,
    event_datetime,
    questionnaire_response_id,
    observation_id
"""


export_sql = f"""
    EXPORT DATA OPTIONS(
      uri='{GCS_DIR}',
      format='PARQUET',
      compression='SNAPPY',
      overwrite=true
    ) AS
    {sql}
    """

job = client.query(export_sql)
job.result()

In [ ]:
from pathlib import Path
import pandas as pd

list_dfs = []

for file in Path(
    "/home/jupyter/workspace/data_bucket/survey_data"
).glob("part-*.parquet"):
    df = pd.read_parquet(
        file,
        columns=["person_id", "llm_event_text"]
    )
    list_dfs.append(df)

df_surveys = pd.concat(list_dfs, ignore_index=True)

df_surveys = (
    df_surveys
    .dropna(subset=["llm_event_text"])
    .groupby("person_id", as_index=False)["llm_event_text"]
    .agg(", ".join)
)

In [ ]:
df_surveys.to_parquet("/home/jupyter/workspace/data_bucket/survey_data/survey.parquet")

# Read MEDS Data

In [ ]:
df = pd.read_parquet('/home/jupyter/workspace/data_bucket/MEDS_DATA/MEDS_cohort/data/tuning/1.parquet')

In [ ]:
df_con = pd.read_parquet('/home/jupyter/workspace/data_bucket/MEDS_DATA/pre_MEDS/procedure_occurrence.parquet')
df_con.columns

In [ ]:
df_codes = pd.read_parquet('/home/jupyter/workspace/data_bucket/MEDS_DATA/MEDS_cohort/metadata/codes.parquet')
df_codes[df_codes['concept_id'].isna()==False].groupby('table_name')['table_name'].agg(lambda x: len(x))
df_codes.groupby("table_name").agg(
    n_codes=("code", "count"),
    n_with_concept_id=("concept_id", lambda x: x.notna().sum()),
    pct_with_concept_id=("concept_id", lambda x: x.notna().mean())
)

In [ ]:
df['table_name'].unique()

In [ ]:
df_proc_oc = pd.read_parquet('/home/jupyter/workspace/data_bucket/MEDS_DATA/pre_MEDS/procedure_occurrence.parquet')
# df_drug_ex = pd.read_parquet('/home/jupyter/workspace/data_bucket/MEDS_DATA/pre_MEDS/drug_exposure.parquet')

In [ ]:
df_proc_oc[df_proc_oc['concept_code']=='01920']

In [ ]:
df_proc_oc[df_proc_oc['concept_code']=='01920']

In [ ]:
np.set_printoptions(suppress=True)
df.head(50)
int(df.loc[26, 'visit_occurrence_id'])

In [ ]:
medred_concept = pd.read_csv('/home/jupyter/workspace/data_bucket/MedRep/concept_idx.csv',low_memory=False)
medred_concept

In [ ]:
medrep = np.load('/home/jupyter/workspace/data_bucket/MedRep/concept_representation_medrep.npy')
medrep

In [ ]:
!mv /home/jupyter/concept_representation_medrep.npy /home/jupyter/workspace/data_bucket/MedRep/